In [15]:
from dotenv import load_dotenv
from langchain_ollama import ChatOllama
load_dotenv()

from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.prompts import (SystemMessagePromptTemplate, HumanMessagePromptTemplate, ChatPromptTemplate)
from langchain_core.output_parsers import StrOutputParser

base_url = "http://localhost:11434"
model = "qwen3:latest"

llm = ChatOllama(model=model, base_url=base_url)

In [16]:
chat_prompt_template = ChatPromptTemplate.from_template("{prompt}?")
chain = chat_prompt_template | llm | StrOutputParser()
chain.invoke({"prompt": "What is my name?"})


"I don't have access to personal information about users, so I can't know your name. However, my name is **Qwen** (通义千问), and I'm here to help with any questions or tasks you need! 😊  \nIf you have other questions, feel free to ask!"

In [17]:
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import SQLChatMessageHistory

In [18]:
def get_session_history(session_id: str):
    return SQLChatMessageHistory(session_id=session_id, connection_string="sqlite:///chat_history.db")

In [20]:
runnable_with_history = RunnableWithMessageHistory(runnable = chain, get_session_history=get_session_history)
user_id = "user_1234"
about = "My name is Shyam and I am a software developer."
runnable_with_history.invoke(
    [HumanMessage(content=about)],
    config={"configurable": {"session_id": user_id}}
)


"Hello, Shyam! It's great to meet you. Welcome to the world of software development—sounds like an exciting field! How are you doing with your projects, or is there anything specific you'd like to discuss? I'm here to help! 😊"

In [21]:
runnable_with_history.invoke(
    [HumanMessage(content="What is my name?")],
    config={"configurable": {"session_id": user_id}}
)

"Your name is Shyam! 😊 Let me know if you need help with anything related to software development or anything else. I'm here to assist!"

In [ ]:
from langchain_core.prompts import (MessagesPlaceholder)

system_msg = SystemMessagePromptTemplate.from_template("You are a helpful assistant ")
human_msg = HumanMessagePromptTemplate.from_template("{input}")
messages = [system_msg, MessagesPlaceholder(variable_name="history"), human_msg]
prompt = ChatPromptTemplate.from_messages(messages)
chain = prompt | llm | StrOutputParser()

runnable_with_history = RunnableWithMessageHistory(runnable=chain, 
                                                   get_session_history=get_session_history, 
                                                   input_messages_key="input",
                                                   history_messages_key="history")
user_id = "user_124"

def chat_with_llm(session_id: str, user_input: str):
    return runnable_with_history.invoke(
        {"input":user_input},
        config={"configurable": {"session_id": session_id}}
    )

In [23]:
chat_with_llm(user_id, "What is my name?")

"Your name is Shyam! 😊 Let me know if you need help with anything—whether it's coding, debugging, or just chatting. I'm here for you!"

In [24]:
user_id = "user_124"
chat_with_llm(user_id, "What is my name?")

"I don't have access to your name unless you've provided it in our conversation. If you'd like, you can share your name with me, and I'll make sure to use it in our interactions! 😊"